# 🧠 CIRCUIT 2.0 — Custom AI Model Fine-Tuning Pipeline
### Complete QLoRA / Unsloth Fine-Tuning on Custom Criminal Intelligence & Financial Forensic Datasets

This Google Colab notebook trains a custom **Meta Llama-3.1-8B-Instruct** model on your proprietary CIRCUIT 2.0 dataset (`train.jsonl` / `val.jsonl`).

**Trained Intelligence Capabilities:**
1. **Bilingual FIR Extraction & NER** (IPC, BNS 318/61, PMLA, suspect PAN/vehicles)
2. **Structuring & Smurfing Anomaly Detection** (Sub-₹50k PAN evasion under Income Tax Act Sec 139A)
3. **Call Detail Records (CDR) Nocturnal Surveillance** (Off-hours cell tower handshakes)
4. **Syndicate Bottleneck Disruption** (Network articulation points & PageRank)
5. **Section 63 BSA Evidence Certification** (Cryptographic SHA-256 electronic evidence certificates)

### Step 1: Select GPU Runtime & Install Dependencies
In Colab menu: Click **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** (Free) or **A100 GPU**.

In [ ]:
!pip install -q --upgrade pip
!pip install -q torch transformers peft trl bitsandbytes datasets accelerate
import torch
print(f"[+] GPU Available: {torch.cuda.is_available()} - Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

### Step 2: Clone CIRCUIT 2.0 Repository & Verify Dataset

In [ ]:
!git clone https://github.com/ei-sanu/circuit2.0.git
%cd circuit2.0
!python scripts/export_finetune_dataset.py

### Step 3: Run QLoRA Fine-Tuning (4-Bit Quantization + LoRA Adapters)

In [ ]:
import os
import urllib.request
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer

BASE_MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"  # Pre-quantized for fast free T4 training
OUTPUT_DIR = "checkpoints/circuit_llama3_lora"

print(f"[*] Loading base model: {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)

# Automatically resolve or download dataset
os.makedirs("data", exist_ok=True)
train_file = "data/train.jsonl"
val_file = "data/val.jsonl"

if not os.path.exists(train_file):
    if os.path.exists("circuit2.0/data/train.jsonl"):
        train_file = "circuit2.0/data/train.jsonl"
        val_file = "circuit2.0/data/val.jsonl"
    elif os.path.exists("/content/circuit2.0/data/train.jsonl"):
        train_file = "/content/circuit2.0/data/train.jsonl"
        val_file = "/content/circuit2.0/data/val.jsonl"
    else:
        print("[*] Downloading dataset directly from GitHub repo...")
        urllib.request.urlretrieve("https://raw.githubusercontent.com/ei-sanu/circuit2.0/main/data/train.jsonl", "data/train.jsonl")
        urllib.request.urlretrieve("https://raw.githubusercontent.com/ei-sanu/circuit2.0/main/data/val.jsonl", "data/val.jsonl")
        train_file = "data/train.jsonl"
        val_file = "data/val.jsonl"

print(f"[+] Loaded Train dataset: {train_file}")
print(f"[+] Loaded Val dataset: {val_file}")

train_ds = load_dataset("json", data_files=train_file)["train"]
val_ds = load_dataset("json", data_files=val_file)["train"]

def format_prompts(batch):
    texts = []
    for inst, inp, out in zip(batch["instruction"], batch["input"], batch["output"]):
        text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou are CIRCUIT AI, a specialized criminal intelligence and financial forensic analysis copilot.<|eot_id|><|start_header_id|>user<|end_header_id|>\n{inst}\n\nContext:\n{inp}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n{out}<|eot_id|>"
        texts.append(text)
    return {"text": texts}

train_ds = train_ds.map(format_prompts, batched=True)
val_ds = val_ds.map(format_prompts, batched=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args,
    peft_config=peft_config,
)

print("[*] Starting QLoRA Fine-Tuning on GPU...")
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n[+] Training complete! Fine-tuned adapters saved to {OUTPUT_DIR}")

### Step 4: Test Real-Time Interactive Inference (ChatGPT-Style)
Ask your fine-tuned model any investigative or natural language query:

In [ ]:
from transformers import pipeline

def ask_circuit_model(user_query, context=""):
    prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou are CIRCUIT AI, an expert criminal intelligence copilot.<|eot_id|><|start_header_id|>user<|end_header_id|>\n{user_query}\n\nContext:\n{context}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=350, temperature=0.2, top_p=0.9)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# Test Query 1: Structuring & Smurfing
print("--- TEST 1: SMURFING DETECTION ---")
print(ask_circuit_model(
    user_query="Analyze the financial transaction chain and check for anti-money laundering anomalies.",
    context="Account: ACC-918237 (Rajesh Verma)\nDeposits: Received ₹48,500 and ₹49,000 within 3 hours from 2 distinct accounts; immediately routed ₹95,000 to Apex Factoring LLC."
))

# Test Query 2: Syndicate Bottleneck
print("\n--- TEST 2: BOTTLENECK DISRUPTION ---")
print(ask_circuit_model(
    user_query="Identify the primary bottleneck node and suggest a disruption strategy.",
    context="Syndicate: Operation Nightfall\nArticulation Points: [P-0001: Rajesh Verma]\nBetweenness Centrality: 0.78 (Connecting Mumbai Port Logistics to Dubai Vaults)"
))

### Step 5: Download Trained Model Weights to Your Local Machine
Run this cell to zip and download your trained adapter weights to drop into CIRCUIT 2.0.

In [ ]:
!zip -r circuit_model_lora.zip checkpoints/circuit_llama3_lora
from google.colab import files
files.download('circuit_model_lora.zip')
print("[+] Download initiated! Extract circuit_model_lora.zip into your local CIRCUIT-2.0/checkpoints/ directory.")